# 03 · First SFT run

This notebook trains our first model with **supervised fine-tuning (SFT)**. The model sees a board position and learns to answer with the move a strong player made there. We train a small **LoRA adapter** rather than the whole model, using the settings in `configs/sft.yaml`.

Then it grades the trained model exactly as `02_eval_base_model.ipynb` graded the base model, and compares the two.

**Before running:** run `01_data_pipeline.ipynb` (it creates the training data) and `02_eval_base_model.ipynb` (it creates the baseline scores).

**Steps**
1. A short **trial run** (500 examples) to check that training works and the loss goes down
2. The **real run** (20,000 examples)
3. **Grading** on the dev and test puzzles
4. **Comparison** with the base model: table and figures

**Outputs**
- the adapter in `checkpoints/sft_v1/` (not on GitHub)
- the run logs `results/sft_v1_training.json`, `results/sft_v1_dev.json`, and `results/sft_v1.json`
- figures in `results/figures/`
- the comparison table `results/progress_report_table.md`

In [ ]:
# Settings
RUN_NAME = "sft_v1"
PUZZLE_LIMIT = None   # None = all puzzles. Set a small number (e.g. 5) for a quick check.

import json

import pandas as pd
import torch

from chesslm.config import REPO_ROOT, load_config
from chesslm.engine import MoveScorer
from chesslm.evaluate import evaluate, load_eval_model, save_run, summarize

model_cfg = load_config("sft")["model"]
prompt_cfg = load_config("prompt")
eval_cfg = load_config("eval")
from chesslm import plots
from chesslm.train_sft import make_dataset, train

sft_cfg = load_config("sft")
SEED = sft_cfg["training"]["seed"]
ADAPTER_DIR = REPO_ROOT / "checkpoints" / RUN_NAME
FIGURES = REPO_ROOT / "results" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)


def run_eval(model, tokenizer, split, run_name, adapter=None):
    """Grade the model on one puzzle set, save the results, and return the summary."""
    puzzles = pd.read_csv(REPO_ROOT / "data" / "eval" / f"puzzles_{split}.csv")
    if PUZZLE_LIMIT:
        puzzles = puzzles.head(PUZZLE_LIMIT)
    with MoveScorer(eval_cfg["stockfish"]) as scorer:
        graded = evaluate(model, tokenizer, puzzles, prompt_cfg, eval_cfg["generation"], scorer)
    summary = summarize(graded)
    config = {"model": model_cfg, "adapter": adapter, "prompt": prompt_cfg, "eval": eval_cfg, "split": split}
    save_run(run_name, graded, summary, config)
    print(f"{run_name}: {len(puzzles)} puzzles | accuracy {summary['puzzle_accuracy']:.1%} | "
          f"legal {summary['legal_rate']:.1%} | ACPL {summary['acpl'] if summary['acpl'] is not None else float('nan'):.0f} | "
          f"{summary['seconds_per_position']:.2f} s per puzzle")
    return graded, summary

## 1. Training data

We sample training and validation examples from the positions built in `01_data_pipeline.ipynb`. Each example is the same prompt used in evaluation, plus the move that was played. The model is trained on the answer only; the prompt is not part of the loss.

In [ ]:
train_positions = pd.read_parquet(REPO_ROOT / "data" / "processed" / "train.parquet")
val_positions = pd.read_parquet(REPO_ROOT / "data" / "processed" / "val.parquet")
n = sft_cfg["data"]

trial_ds = make_dataset(train_positions, n["trial_examples"], prompt_cfg, SEED)
train_ds = make_dataset(train_positions, n["train_examples"], prompt_cfg, SEED)
val_ds = make_dataset(val_positions, n["val_examples"], prompt_cfg, SEED)
print(f"trial: {len(trial_ds):,}  train: {len(train_ds):,}  validation: {len(val_ds):,}\n")

example = train_ds[0]
print("Prompt:    ", example["prompt"][0]["content"].replace("\n", "\n            "))
print("Completion:", example["completion"][0]["content"])

## 2. Trial run (500 examples)

A quick check that everything works on this machine. The loss should go down.

In [ ]:
trial_log, trial_compute = train(trial_ds, val_ds.select(range(min(100, len(val_ds)))), sft_cfg, REPO_ROOT / "checkpoints" / "sft_trial")
losses = trial_log.loss.dropna()
print(f"\nTraining loss: {losses.iloc[0]:.3f} at the start -> {losses.iloc[-1]:.3f} at the end")
print(trial_compute)
torch.cuda.empty_cache()

## 3. Real run (20,000 examples)

This is the main training run. It takes the longest. Progress is shown below as it runs.

In [ ]:
log, compute = train(train_ds, val_ds, sft_cfg, ADAPTER_DIR)

with open(REPO_ROOT / "results" / f"{RUN_NAME}_training.json", "w") as f:
    json.dump({"config": sft_cfg, "prompt": prompt_cfg, "compute": compute,
               "log": log.to_dict("records")}, f, indent=2, default=str)
print(json.dumps(compute, indent=2))
plots.loss_curve(log, FIGURES / f"{RUN_NAME}_loss.png");

## 4. Grade the trained model

The same grading code, puzzles, and settings as the base model. Only the model differs: the base model plus our trained adapter.

In [ ]:
import gc

gc.collect()  # free the memory used by training before loading the model for grading
torch.cuda.empty_cache()
tokenizer, model = load_eval_model(model_cfg, adapter_path=ADAPTER_DIR)
dev_graded, dev_summary = run_eval(model, tokenizer, "dev", f"{RUN_NAME}_dev", adapter=str(ADAPTER_DIR))
test_graded, test_summary = run_eval(model, tokenizer, "test", RUN_NAME, adapter=str(ADAPTER_DIR))

## 5. Compare with the base model

All numbers below are on the 1,000-puzzle test set.

In [ ]:
summary = pd.read_csv(REPO_ROOT / "results" / "summary.csv")
runs = ["base", RUN_NAME]
table = summary.set_index("run").loc[runs]

columns = {"puzzle_accuracy": "Accuracy", "accuracy_<1200": "<1200", "accuracy_1200-1800": "1200-1800",
           "accuracy_1800-2400": "1800-2400", "accuracy_2400+": "2400+", "legal_rate": "Legal",
           "illegal_rate": "Illegal", "unparseable_rate": "Unparseable", "acpl": "ACPL"}
report = table[list(columns)].rename(columns=columns)
percent = [c for c in report.columns if c != "ACPL"]
report[percent] = (report[percent] * 100).round(1)
report["ACPL"] = report["ACPL"].round(0)
report = report.astype(object).where(report.notna(), "–")  # e.g. ACPL when there are no legal moves

# Save as a Markdown table for the progress report (numbers only).
lines = ["| Model | " + " | ".join(report.columns) + " |", "|---" * (len(report.columns) + 1) + "|"]
lines += [f"| {run} | " + " | ".join(str(v) for v in row) + " |" for run, row in report.iterrows()]
note = "\n\nTest set: 1,000 Lichess puzzles (250 per rating band). Accuracy, legal, illegal, unparseable in %; ACPL in centipawns on legal moves."
(REPO_ROOT / "results" / "progress_report_table.md").write_text("\n".join(lines) + note + "\n")
report

In [ ]:
plots.compare_metrics(summary, runs, FIGURES / "base_vs_sft_metrics.png");
plots.accuracy_by_band(summary, runs, FIGURES / "accuracy_by_band.png");
plots.outcome_breakdown(summary, runs, FIGURES / "answer_outcomes.png");

## Summary

The trained adapter is in `checkpoints/sft_v1/`. The scores, training log, figures, and comparison table are in `results/`.

**After running:** push `results/` and `notebooks/` to GitHub (see the README), and share `checkpoints/sft_v1/` with the team. DPO and RL start from this model.